# Experiment 06: Binary Classification using Logistic Regression


### 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    accuracy_score
)

print("Libraries imported successfully.")


### 2. Load and Explore the Dataset (Recode Malignant=1, Benign=0)


In [ ]:
dataset = load_breast_cancer(as_frame=True)

X = dataset.data.copy()
# Recode so positive class is malignant (1 = malignant, 0 = benign)
y = (dataset.target == 0).astype(int)
y.name = "malignant"

df = X.copy()
df["target"] = y

print("Dataset shape (samples, features):", X.shape)
print("
First five rows:")
display(df.head())

print("
Class counts (0 = benign, 1 = malignant):")
print(y.value_counts().sort_index().rename(index={0: "Benign (0)", 1: "Malignant (1)"}))

print("
Missing values in feature columns:", int(X.isnull().sum().sum()))


### 3. Prepare Training and Testing Data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])
print("
Training class distribution:")
print(y_train.value_counts().sort_index().rename(index={0: "Benign (0)", 1: "Malignant (1)"}))
print("
Testing class distribution:")
print(y_test.value_counts().sort_index().rename(index={0: "Benign (0)", 1: "Malignant (1)"}))


### 4. Build and Train Logistic Regression Pipeline


In [ ]:
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=42))
])

model.fit(X_train, y_train)
print("Logistic Regression model trained successfully.")


### 5. Predict Test-Set Labels


In [ ]:
y_pred = model.predict(X_test)

print("First 20 actual labels   :", y_test.to_numpy()[:20])
print("First 20 predicted labels :", y_pred[:20])
print("Test accuracy: {:.4f}".format(accuracy_score(y_test, y_pred)))


### 6. Confusion Matrix


In [ ]:
labels = [0, 1]
cm = confusion_matrix(y_test, y_pred, labels=labels)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print(f"True Negatives (TN) : {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP) : {tp}")

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Benign (0)", "Malignant (1)"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Logistic Regression - Confusion Matrix")
plt.tight_layout()
plt.show()


### 7. Evaluation Metrics: Precision, Recall, and F1-score


In [ ]:
precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

metrics = pd.DataFrame({
    "Metric": ["Precision (malignant)", "Recall (malignant)", "F1-score (malignant)", "Accuracy"],
    "Value": [precision, recall, f1, accuracy_score(y_test, y_pred)]
})

display(metrics.style.format({"Value": "{:.4f}"}))

print("
Classification report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Benign (0)", "Malignant (1)"],
    digits=4,
    zero_division=0
))


### 8. Visualize Evaluation Metrics


In [ ]:
metric_names = ["Precision", "Recall", "F1-score"]
metric_values = [precision, recall, f1]

plt.figure(figsize=(7, 4))
bars = plt.bar(metric_names, metric_values, color=["#1f77b4", "#2ca02c", "#d62728"])
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Performance Metrics for Positive Class: Malignant")
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bar, value in zip(bars, metric_values):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.02,
             f"{value:.3f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()
